In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'HOUR/DATE',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

#### Load data

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()

In [ ]:
df = df.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df['HOUR/DATE'] = pd.to_datetime(df['HOUR/DATE'])

df['WEEKDAY'] = df['HOUR/DATE'].dt.dayofweek  # Extract weekday (Monday=0, Sunday=6)
df['HOUR'] = df['HOUR/DATE'].dt.hour

df = df.drop(columns=['HOUR/DATE'])

In [ ]:
df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

In [ ]:
df

In [ ]:
negative_policies = df[df['FLOW'] < 0]['POLICY'].values
filtered_df = df[~df['POLICY'].isin(negative_policies)]

filtered_df

In [ ]:
filtered_df.loc[(filtered_df['USAGE'] == 'AJUNTAMENT') & (filtered_df['FLOW'] > 4000)] 

In [ ]:
ajuntament_flow_greater_than_4000 = filtered_df.loc[(filtered_df['USAGE'] == 'AJUNTAMENT') & (filtered_df['FLOW'] > 4000)]['POLICY'].values
new_filtered_df = filtered_df[~filtered_df['POLICY'].isin(ajuntament_flow_greater_than_4000)]

new_filtered_df

In [ ]:
new_filtered_df['WEEKDAY'].unique()

#### Calculate the mean flow for each hour, and plot it

In [ ]:
average_per_hour_usage = new_filtered_df.groupby(['USAGE', 'HOUR'])['FLOW'].mean().reset_index()

In [ ]:
# Assuming you have a DataFrame 'average_per_hour_usage' with columns 'HOUR', 'USAGE', and 'FLOW'
usage_types = average_per_hour_usage['USAGE'].unique()

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i, usage in enumerate(usage_types):
  data = average_per_hour_usage[average_per_hour_usage['USAGE'] == usage]
  ax = axs[i // 2, i % 2]  # Assign subplot to each usage type
  ax.bar(data['HOUR'], data['FLOW'])  # Use bar function for bar graph
  ax.set_xlabel('Hour')
  ax.set_ylabel('Flow')
  ax.set_title(f'Flow for {usage}')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from scipy import stats
import numpy as np

def label_data_with_gaussian_means(df, threshold=2):
    """
    Labels data as 'leak' or 'no leak' based on Gaussian means and z-scores.

    Args:
      df: Pandas DataFrame with 'CONSUMPTION', 'POLICY', 'WEEKDAY', 'HOUR',
          'USAGE', and 'HOUSING' columns.
      threshold: Z-score threshold for identifying anomalies.

    Returns:
      Pandas DataFrame with an additional 'LEAK' column (True for leak, 
      False for no leak).
    """

    # Apply your filters here
    df = df.rename(columns=short_names)
    df['HOUR/DATE'] = pd.to_datetime(df['HOUR/DATE'])
    df['WEEKDAY'] = df['HOUR/DATE'].dt.dayofweek
    df['HOUR'] = df['HOUR/DATE'].dt.hour
    df = df.drop(columns=['HOUR/DATE'])
    df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
    df = df.dropna(subset=["FLOW"])
    negative_policies = df[df['FLOW'] < 0]['POLICY'].values
    df = df[~df['POLICY'].isin(negative_policies)]
    ajuntament_flow_greater_than_4000 = df.loc[(df['USAGE'] == 'AJUNTAMENT') & (df['FLOW'] > 4000)]['POLICY'].values
    df = df[~df['POLICY'].isin(ajuntament_flow_greater_than_4000)]

    # Group data by relevant features (excluding 'HOUSING')
    grouped = df.groupby(['POLICY', 'WEEKDAY', 'HOUR', 'USAGE', 'HOUSING'])  

    # Calculate mean and standard deviation for each group
    def calculate_stats(group):
        return pd.Series({
            'mean_consumption': group['CONSUMPTION'].mean(),
            'std_consumption': group['CONSUMPTION'].std()
        })

    group_stats = grouped.apply(calculate_stats).reset_index()

    # Merge group statistics back into the original DataFrame
    df = df.merge(group_stats, on=['POLICY', 'WEEKDAY', 'HOUR', 'USAGE'])  

    # Calculate z-scores
    df['zscore'] = (df['CONSUMPTION'] - df['mean_consumption']) / df['std_consumption']

    # Replace infinite values with NaN
    df['zscore'].replace([np.inf, -np.inf], np.nan, inplace=True)

    # Fill NaN values with 0 (or another appropriate value)
    df['zscore'].fillna(0, inplace=True)

    # Label data based on z-score threshold
    df['LEAK'] = df['zscore'] > threshold

    return df

# Assuming you have a DataFrame 'df' with the necessary columns

# Label the data
df_labeled = label_data_with_gaussian_means(df, threshold=2.5)

# Print the labeled DataFrame
print(df_labeled.head())

In [ ]:
df_labeled